# 08 · Learning curves

Trains one model on growing parts of the training set (1% → 100%) and measures validation accuracy each time. Pick the model in `MODEL_KEY`, run all cells, then switch to the next one. The last cell draws all curves in one chart.

The subsets are nested (each one contains the smaller ones) and keep the three classes balanced. The test set isn't used.

In [ ]:
%pip install -q -U transformers datasets accelerate scikit-learn matplotlib python-dotenv

## Settings

In [ ]:
import gc
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

# ---- Pick the model ----
MODEL_KEY = "tfidf"

MODELS = {
    "tfidf": None,                                                    # runs on a CPU in a few minutes
    "distilbert-german": "distilbert/distilbert-base-german-cased",
    "mmbert-small": "jhu-clsp/mmBERT-small",
    "xlm-roberta-base": "FacebookAI/xlm-roberta-base",
    "mmbert-base": "jhu-clsp/mmBERT-base",
    "gelectra-large": "deepset/gelectra-large",
}

FRACTIONS = [0.01, 0.05, 0.12, 0.175, 0.25, 0.50, 1.00]
SEED = 17
MAX_LENGTH = 512
EPOCHS = 3            # same as step 07, but no best-epoch selection: the model after the last epoch is used
LEARNING_RATE = 2e-5
BATCH_SIZE = 8

HF_DATASET_REPO = "aliissa824/german-academic-ai-text"
RESULTS_FILE = Path("results") / "learning-curves.csv"
RESULTS_FILE.parent.mkdir(exist_ok=True)

load_dotenv(Path.cwd().parent / ".env")
HF_TOKEN = os.getenv("HF_TOKEN")

## Data and nested, balanced subsets

In [ ]:
dataset = load_dataset(HF_DATASET_REPO, token=HF_TOKEN)
train, validation = dataset["train"], dataset["validation"]


def balanced_order(labels, seed):
    """Shuffle each class, then interleave them so every prefix keeps the class balance.
    Taking the first n texts of this order gives nested subsets."""
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    pools = {c: rng.permutation(np.flatnonzero(labels == c)).tolist() for c in np.unique(labels)}
    share = {c: len(p) / len(labels) for c, p in pools.items()}
    taken = {c: 0 for c in pools}
    order = []
    for step in range(len(labels)):
        c = max((c for c in pools if taken[c] < len(pools[c])), key=lambda c: share[c] * (step + 1) - taken[c])
        order.append(pools[c][taken[c]])
        taken[c] += 1
    return np.array(order)


order = balanced_order(train["label"], SEED)
sizes = [int(round(f * len(train))) for f in FRACTIONS]
print("Subset sizes:", sizes)

## Train on each subset

In [ ]:
def run_tfidf(indices):
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), max_features=200_000, min_df=5, max_df=0.98,
                                          sublinear_tf=True, dtype=np.float32),
                          LogisticRegression(C=3.0, max_iter=1000, random_state=SEED))
    texts, labels = np.array(train["text"], dtype=object)[indices], np.array(train["label"])[indices]
    model.fit(texts, labels)
    return accuracy_score(labels, model.predict(texts)), accuracy_score(validation["label"], model.predict(validation["text"]))


def run_transformer(indices, tokenized, tokenizer):
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODELS[MODEL_KEY], num_labels=3)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
    args = TrainingArguments(
        output_dir=f"checkpoints/curve-{MODEL_KEY}", learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE * 2, weight_decay=0.01,
        eval_strategy="no", save_strategy="no", logging_strategy="epoch", report_to="none",
        bf16=use_bf16, fp16=torch.cuda.is_available() and not use_bf16, seed=SEED, data_seed=SEED)
    subset = tokenized["train"].select(indices.tolist())
    trainer = Trainer(model=model, args=args, train_dataset=subset, processing_class=tokenizer,
                      data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    accuracy = lambda part: accuracy_score(part["label"], trainer.predict(part).predictions.argmax(-1))
    result = accuracy(subset), accuracy(tokenized["validation"])
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return result


if MODEL_KEY != "tfidf":
    tokenizer = AutoTokenizer.from_pretrained(MODELS[MODEL_KEY])
    tokenized = dataset.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True)
    tokenized = tokenized.select_columns(["input_ids", "attention_mask", "label"])

rows = []
for size in sizes:
    start = time.time()
    indices = order[:size]
    train_acc, val_acc = run_tfidf(indices) if MODEL_KEY == "tfidf" else run_transformer(indices, tokenized, tokenizer)
    rows.append({"model": MODEL_KEY, "training_texts": size, "training_fraction": round(size / len(train), 4),
                 "training_accuracy": round(train_acc, 4), "validation_accuracy": round(val_acc, 4)})
    print(f"{size:>6,} texts: train {train_acc:.1%}, validation {val_acc:.1%} ({time.time() - start:.0f}s)")

## Save

Replaces this model's rows in `results/learning-curves.csv` and keeps the other models.

In [ ]:
new = pd.DataFrame(rows)
if RESULTS_FILE.exists():
    old = pd.read_csv(RESULTS_FILE)
    new = pd.concat([old[old["model"] != MODEL_KEY], new])
new.to_csv(RESULTS_FILE, index=False)
print(new.pivot(index="training_texts", columns="model", values="validation_accuracy").to_string())

## Chart of all models

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

d = pd.read_csv(RESULTS_FILE)
SERIES = [  # (key, label, color, marker, style)
    ("gelectra-large", "GELECTRA-large", "#2a78d6", "o", "-"),
    ("mmbert-base", "mmBERT-base", "#eb6834", "s", "-"),
    ("mmbert-small", "mmBERT-small", "#1baf7a", "D", "-"),
    ("xlm-roberta-base", "XLM-RoBERTa base", "#eda100", "^", "-"),
    ("distilbert-german", "DistilBERT German", "#e87ba4", "v", "-"),
    ("tfidf", "TF-IDF", "#6b6a66", "o", "--"),
]
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
fig, ax = plt.subplots(figsize=(9.2, 5.6), dpi=160)
fig.patch.set_facecolor("#fcfcfb"); ax.set_facecolor("#fcfcfb")
ends = []
for key, label, color, marker, ls in SERIES:
    p = d[d.model == key].sort_values("training_texts")
    if p.empty:
        continue
    ax.plot(p.training_texts, p.validation_accuracy * 100, ls, color=color, lw=2, zorder=3)
    ax.scatter(p.training_texts, p.validation_accuracy * 100, s=42, marker=marker, color=color,
               edgecolor="#fcfcfb", linewidth=1.6, zorder=4)
    ends.append([p.validation_accuracy.iloc[-1] * 100, label, color])
# direct labels at the right end, nudged apart
ends.sort(key=lambda e: -e[0])
pos = [e[0] for e in ends]
gap = 2.6
for i in range(1, len(pos)):
    pos[i] = min(pos[i], pos[i - 1] - gap)
x_end, x_lab = 24986, 24986 * 1.25
for (y, label, color), yl in zip(ends, pos):
    ax.plot([x_end * 1.04, x_lab * 0.97], [y, yl], color="#b8b7b1", lw=0.8, zorder=2)
    ax.text(x_lab, yl, f"{label}  {y:.1f}%", va="center", ha="left", fontsize=10, color="#0b0b0b")
ax.set_xscale("log")
ax.set_xlim(200, 24986 * 3.6)
ax.set_xticks([250, 1249, 2998, 6246, 12493, 24986])
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{int(round(v)):,}"))
ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
ax.set_ylim(30, 100)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.grid(axis="y", color="#e6e5e0", lw=0.8); ax.set_axisbelow(True)
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#b8b7b1")
ax.tick_params(colors="#52514e", length=0)
ax.set_xlabel("Training texts (log scale)", color="#52514e")
ax.set_ylabel("Validation accuracy", color="#52514e")
ax.axhline(100 / 3, color="#b8b7b1", lw=0.8, ls=":")
ax.text(210, 100 / 3 + 1, "guessing (33%)", fontsize=9, color="#52514e")
ax.set_title("Accuracy vs. amount of training data", loc="left", fontsize=13, color="#0b0b0b", pad=12)
fig.tight_layout()
fig.savefig(RESULTS_FILE.with_suffix(".png"), facecolor=fig.get_facecolor())
plt.show()